# 61 — Verbal aggression: the irtree workflow (paper Section 6.1)

This notebook is the code of Section 6.1. It fits De Boeck and Partchev's (2012) linear tree to the verbal aggression data, step by step:

1. read the data and declare the tree,
2. prepare the data with person and item covariates,
3. choose the node models and look at the priors,
4. fit, check convergence, interpret,
5. convert to the logistic metric, and
6. extend the nodes from Rasch to 2PL and compare the two models.

**Data.** `verbagg_long.csv` is lme4's `VerbAgg`: 316 persons, 24 items (*"A bus fails to stop for me. I would want to curse"*), responses no / perhaps / yes, with person covariates (Anger score, gender) and item covariates (behavior type: curse, scold, shout; situation: other or self to blame; mode: want or do).

**Runtime.** Each fit is timed; the last cell saves the table to `results/runtime_61.csv`.

## Setup

In [1]:
# Install the package from this repository (editable), once per environment.
# The notebook lives in replication/<folder>/, so the repository root is two levels up.
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
!{sys.executable} -m pip install -q -e "{ROOT}[full]"

In [2]:
import warnings
import numpy as np, pandas as pd
import irtree
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

# Runtime log: wrap every expensive step in `with timed("label"):`.
# The last cell prints the table and saves it to results/.
import time, contextlib
RUNTIME = []

@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    RUNTIME.append({"step": step, "seconds": round(dt, 1)})
    print(f"[{step}] " + (f"{dt / 60:.1f} min" if dt >= 60 else f"{dt:.1f} s"))

## Settings
Four chains run in parallel processes. `N_ITER` counts warm-up and retained draws; with the default evidence-based warm-up, the retained draws are reported by the fit.

In [3]:
N_ITER   = 4000
N_CHAINS = 4
SEED     = 2026
print(f"N_ITER={N_ITER}  N_CHAINS={N_CHAINS}  SEED={SEED}")

N_ITER=4000  N_CHAINS=4  SEED=2026


## Step 1 — Read the data
One row per person × item. The responses are recoded to 0 (no), 1 (perhaps), 2 (yes).

In [4]:
va = pd.read_csv("verbagg_long.csv")
va["response"] = va["resp"].map({"no": 0, "perhaps": 1, "yes": 2})
print(va.shape)
print(va["resp"].value_counts())
va.head()

(7584, 9)
resp
no         3973
perhaps    2081
yes        1530
Name: count, dtype: int64


,id,item,resp,Anger,Gender,btype,situ,mode,response
0,1,S1WantCurse,no,20,M,curse,other,want,0
1,2,S1WantCurse,no,11,M,curse,other,want,0
2,3,S1WantCurse,perhaps,17,F,curse,other,want,1
3,4,S1WantCurse,perhaps,21,F,curse,other,want,1
4,5,S1WantCurse,perhaps,17,F,curse,other,want,1


## Step 2 — Covariates
All covariates are centered (Section 4: the traits and locations have fixed means, so uncentered covariates would shift them).

* person: `anger` (Anger score) and `male` (gender indicator);
* item: `scold`, `shout` (curse is the reference), `self` (self to blame), `do` (doing rather than wanting), each a centered indicator, plus `intercept`, a column of ones. Because the item locations have prior mean 0, the intercept's coefficient is the node's average location shift: the node intercept.

In [5]:
def centered(x):
    x = x.astype(float)
    return x - x.mean()

pers = va.drop_duplicates("id")
items = va.drop_duplicates("item")
person_cov = pd.DataFrame({"id": pers["id"],
                           "anger": centered(pers["Anger"]),
                           "male":  centered(pers["Gender"] == "M")})
item_cov = pd.DataFrame({"item": items["item"],
                         "intercept": 1.0,
                         "scold": centered(items["btype"] == "scold"),
                         "shout": centered(items["btype"] == "shout"),
                         "self":  centered(items["situ"] == "self"),
                         "do":    centered(items["mode"] == "do")})
table = va[["id", "item", "response"]].merge(person_cov, on="id").merge(item_cov, on="item")
X_NAMES = ["anger", "male"]
Q_NAMES = ["intercept", "scold", "shout", "self", "do"]
table.head()

,id,item,response,anger,male,intercept,scold,shout,self,do
0,1,S1WantCurse,0,-0.003165,0.768987,1.0,-0.333333,-0.333333,-0.5,-0.5
1,2,S1WantCurse,0,-9.003165,0.768987,1.0,-0.333333,-0.333333,-0.5,-0.5
2,3,S1WantCurse,1,-3.003165,-0.231013,1.0,-0.333333,-0.333333,-0.5,-0.5
3,4,S1WantCurse,1,0.996835,-0.231013,1.0,-0.333333,-0.333333,-0.5,-0.5
4,5,S1WantCurse,1,-3.003165,-0.231013,1.0,-0.333333,-0.333333,-0.5,-0.5


## Step 3 — Declare the tree
The linear tree has two nodes. Node 1 separates *no* from *perhaps or yes*; node 2, reached only by *perhaps or yes*, separates *perhaps* from *yes*. Rows of the mapping matrix are the categories 0, 1, 2; `NaN` marks a node that is not on the path.

In [7]:
spec = irtree.IRTreeSpec(mapping=np.array([[0, np.nan],    # no
                                           [1, 0],         # perhaps
                                           [1, 1]]),       # yes
                         category_names=("no", "perhaps", "yes"),
                         node_names=("respond", "yes"))
spec

IRTreeSpec(K=3, R=2, latent_dims=2)
  Categories:        K = 3
  Nodes:             R = 2
  Latent dimensions: 2
  Outcomes per node: [2, 2]

## Step 4 — Prepare the data
`load_data` in single-table mode: the long table carries the responses and both kinds of covariates; `roles` says which columns are person (X) and item (Q) covariates. It aligns everything by person and item id and expands the responses into the pseudo-item responses of each node.

In [10]:
data = irtree.load_data(table, spec=spec, person_id="id", item_id="item",
                        response_col="response", roles={"X": X_NAMES, "Q": Q_NAMES},
                        categories_start=0, center = False, scale = False)


print(irtree.summarize_prepared(data, spec))

Ingest: P=316 persons, I=24 items (first-seen order).
PreparedData: P=316, I=24, R=2, missing=0
  Node 1: n_relevant=7584, mean_Y*=0.476
  Node 2: n_relevant=3611, mean_Y*=0.424
  X (person covariates): K_x=2, n_missing=0
  Q (item covariates): K_q=5, n_missing=0


In [11]:
data

PreparedData(Y_star=[array([[0., 0., 0., ..., 1., 1., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 1., 1., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 1., 0., ..., 1., 1., 0.],
       [1., 1., 1., ..., 0., 0., 0.]], shape=(316, 24)), array([[nan, nan, nan, ...,  1.,  1.,  1.],
       [nan, nan, nan, ..., nan, nan, nan],
       [ 0.,  0.,  0., ...,  0., nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [ 1.,  1., nan, ...,  0.,  0., nan],
       [ 0.,  0.,  0., ..., nan, nan, nan]], shape=(316, 24))], T=[array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]], shape=(316, 24)), array([[False, False, False, ...,  True,  True,  True]

In [9]:
data

PreparedData(Y_star=[array([[0., 0., 0., ..., 1., 1., 1.],
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 1., 1., ..., 1., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [1., 1., 0., ..., 1., 1., 0.],
       [1., 1., 1., ..., 0., 0., 0.]], shape=(316, 24)), array([[nan, nan, nan, ...,  1.,  1.,  1.],
       [nan, nan, nan, ..., nan, nan, nan],
       [ 0.,  0.,  0., ...,  0., nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [ 1.,  1., nan, ...,  0.,  0., nan],
       [ 0.,  0.,  0., ..., nan, nan, nan]], shape=(316, 24))], T=[array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]], shape=(316, 24)), array([[False, False, False, ...,  True,  True,  True]

## Step 5 — Node models and priors (Rasch)
Both nodes are probit Rasch nodes that use every covariate. Under the probit link the sampler draws every block exactly except the trait correlation, which has one Metropolis-Hastings step.

The package defaults are printed by `priors.summary()`. Two changes: the covariate coefficients get a fixed, weakly informative N(0, 10) prior instead of the default hierarchical one, whose shared variance would shrink the intercept and the slopes toward one another; with five item coefficients per node there is too little information to estimate that variance.

In [9]:
nodes_rasch = [irtree.ProbitRaschNode(person_covariates=True, item_covariates=True),
               irtree.ProbitRaschNode(person_covariates=True, item_covariates=True)]

priors_rasch = irtree.build_default_priors(spec, nodes_rasch)

priors_rasch.set("beta",  irtree.NormalPrior(mean=0.0, variance=10.0))
priors_rasch.set("kappa", irtree.NormalPrior(mean=0.0, variance=10.0))
priors_rasch.summary()



TABLE 1 — Main parameters
──────────────────────────────────────────────────────────────────────────────────────────────────────────
Parameter       Node 1                                        Node 2                                      
──────────────────────────────────────────────────────────────────────────────────────────────────────────
theta           Normal(0.0, [estimated~InvGammaPrior] theta.variance) [2]  Normal(0.0, [estimated~InvGammaPrior] theta.variance) [2]
──────────────────────────────────────────────────────────────────────────────────────────────────────────
b               Normal(0.0, [estimated~InvGammaPrior] b.variance) [2]  Normal(0.0, [estimated~InvGammaPrior] b.variance) [2]
──────────────────────────────────────────────────────────────────────────────────────────────────────────
a               —                                             —                                           
─────────────────────────────────────────────────────────────────────────

In [11]:
nodes_rasch_no_cov = [irtree.ProbitRaschNode(),
               irtree.ProbitRaschNode()]

priors_rasch_no_cov = irtree.build_default_priors(spec, nodes_rasch_no_cov)

priors_rasch_no_cov.summary()


TABLE 1 — Main parameters
──────────────────────────────────────────────────────────────────────────────────────────────────────────
Parameter       Node 1                                        Node 2                                      
──────────────────────────────────────────────────────────────────────────────────────────────────────────
theta           Normal(0.0, [estimated~InvGammaPrior] theta.variance) [2]  Normal(0.0, [estimated~InvGammaPrior] theta.variance) [2]
──────────────────────────────────────────────────────────────────────────────────────────────────────────
b               Normal(0.0, [estimated~InvGammaPrior] b.variance) [2]  Normal(0.0, [estimated~InvGammaPrior] b.variance) [2]
──────────────────────────────────────────────────────────────────────────────────────────────────────────
a               —                                             —                                           
─────────────────────────────────────────────────────────────────────────

## Step 6 — Fit

In [14]:
with timed("fit Rasch tree"):
    fit_rasch = irtree.irtree_fit(data, spec, nodes_rasch, priors=priors_rasch,
                                  n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

sample 400/4000 (10%) · 252.0 it/s · elapsed 0:01 · eta 0:14 · loglik -7053.9 · accept 0.39
sample 800/4000 (20%) · 234.7 it/s · elapsed 0:03 · eta 0:13 · loglik -5610.2 · accept 0.42
sample 1200/4000 (30%) · 233.2 it/s · elapsed 0:05 · eta 0:12 · loglik -5577.0 · accept 0.43
sample 1600/4000 (40%) · 230.8 it/s · elapsed 0:06 · eta 0:10 · loglik -5634.0 · accept 0.43
sample 2000/4000 (50%) · 227.7 it/s · elapsed 0:08 · eta 0:08 · loglik -5577.0 · accept 0.43
sample 2400/4000 (60%) · 226.5 it/s · elapsed 0:10 · eta 0:07 · loglik -5613.3 · accept 0.43
sample 2800/4000 (70%) · 225.9 it/s · elapsed 0:12 · eta 0:05 · loglik -5625.0 · accept 0.44
sample 3200/4000 (80%) · 225.9 it/s · elapsed 0:14 · eta 0:03 · loglik -5602.2 · accept 0.44
sample 3600/4000 (90%) · 225.5 it/s · elapsed 0:15 · eta 0:01 · loglik -5623.5 · accept 0.44
sample 4000/4000 (100%) · 225.8 it/s · elapsed 0:17 · eta 0:00 · loglik -5612.1 · accept 0.44
done 4000/4000 (100%) · elapsed 0:17



irtree fit complete  (1 chain × 4000 iterations, 3600 posterior draws)
   parameters: 687 PASS · 12 WARN · 0 FAIL   (of 699)
   1 flagged only for under-warmup — resolve by discarding the first ~10% (recommended_burnin).
   (overall label is advisory; the per-parameter table is the decision. Call fit.diagnostics().)
   status key:
     PASS (converged): stationary with no burn-in needed, adequate mixing.
     WARN (usable, but check): at worst under-warmed (a recommended burn-in is given), OR the two stationarity tests (Geweke & Heidelberger-Welch) disagree (one passes, one flags), OR mixing/precision is marginal (low ESS, high MCSE, R-hat slightly > 1).
     FAIL (did not converge): fails BOTH stationarity tests (Geweke & Heidelberger-Welch), or R-hat exceeds its hard threshold. A parameter that passes EITHER stationarity test is a WARN, not a FAIL.
⚠  Single chain — R-hat unavailable. Stationarity via Geweke + Heidelberger-Welch.
✓ Stationarity (Geweke):   All Geweke within threshol

In [15]:
with timed("fit Rasch tree"):
    fit_rasch_no_cov = irtree.irtree_fit(data, spec, nodes_rasch_no_cov, priors=priors_rasch_no_cov,
                                  n_iter=N_ITER, n_chains=1, seed=SEED, progress=True)

sample 400/4000 (10%) · 434.8 it/s · elapsed 0:00 · eta 0:08 · loglik -7053.9 · accept 0.41
sample 800/4000 (20%) · 408.2 it/s · elapsed 0:01 · eta 0:07 · loglik -5596.1 · accept 0.44
sample 1200/4000 (30%) · 405.1 it/s · elapsed 0:02 · eta 0:06 · loglik -5617.2 · accept 0.43
sample 1600/4000 (40%) · 401.9 it/s · elapsed 0:03 · eta 0:05 · loglik -5616.3 · accept 0.42
sample 2000/4000 (50%) · 403.9 it/s · elapsed 0:04 · eta 0:04 · loglik -5624.2 · accept 0.43
sample 2400/4000 (60%) · 405.3 it/s · elapsed 0:05 · eta 0:03 · loglik -5604.4 · accept 0.43
sample 2800/4000 (70%) · 405.4 it/s · elapsed 0:06 · eta 0:02 · loglik -5598.8 · accept 0.43
sample 3200/4000 (80%) · 405.3 it/s · elapsed 0:07 · eta 0:01 · loglik -5596.5 · accept 0.43
sample 3600/4000 (90%) · 405.5 it/s · elapsed 0:08 · eta 0:00 · loglik -5601.3 · accept 0.42
sample 4000/4000 (100%) · 405.4 it/s · elapsed 0:09 · eta 0:00 · loglik -5595.1 · accept 0.42
done 4000/4000 (100%) · elapsed 0:09



irtree fit complete  (1 chain × 4000 iterations, 3600 posterior draws)
   parameters: 676 PASS · 9 WARN · 0 FAIL   (of 685)
   3 flagged only for under-warmup — resolve by discarding the first ~10% (recommended_burnin).
   (overall label is advisory; the per-parameter table is the decision. Call fit.diagnostics().)
   status key:
     PASS (converged): stationary with no burn-in needed, adequate mixing.
     WARN (usable, but check): at worst under-warmed (a recommended burn-in is given), OR the two stationarity tests (Geweke & Heidelberger-Welch) disagree (one passes, one flags), OR mixing/precision is marginal (low ESS, high MCSE, R-hat slightly > 1).
     FAIL (did not converge): fails BOTH stationarity tests (Geweke & Heidelberger-Welch), or R-hat exceeds its hard threshold. A parameter that passes EITHER stationarity test is a WARN, not a FAIL.
⚠  Single chain — R-hat unavailable. Stationarity via Geweke + Heidelberger-Welch.
✓ Stationarity (Geweke):   All Geweke within threshold

## Step 7 — Convergence
The console summary classifies every parameter; `diagnostics()` returns the full table (R-hat, bulk and tail ESS, MCSE/SD).

In [ ]:
rep = fit_rasch.diagnostics()
rep

## Step 8 — Interpret
The structural parameters: trait variances (estimated on Rasch dimensions), the trait correlation, the item-variance of each node, and the covariate effects. Positive coefficients raise the probability of the branch coded 1 (responding at node 1, saying *yes* at node 2).

In [ ]:
summ = fit_rasch.summary_df()
structural = [n for n in summ.index if n.startswith(("sigma2", "rho", "b.variance", "beta_", "kappa_"))]
summ.loc[structural, ["mean", "sd", "hdi_lo", "hdi_hi", "n_eff"]].round(3)

## Step 9 — The logistic metric
`to_logit()` puts the summary on the logistic-equivalent scale (D = 1.702) so the estimates can be read against logistic results such as those of Section 7.2. On these slope-free dimensions the factor goes on the traits, locations and coefficients; variances take its square; the correlation is unchanged. The conversion is approximate in the tails.

In [ ]:
lg = fit_rasch.to_logit()
print(*lg.attrs.get("notes", []), sep="\n")
lg.loc[structural, ["mean", "sd", "factor"]].round(3)

## Step 10 — From Rasch to 2PL nodes
Replacing the node models is the only change. On these dimensions every node now has slopes, so the trait variances are fixed at 1 (identification, Section 4) and the slopes carry the scale. The correlation is scale-free and stays estimated.

In [ ]:
nodes_2pl = [irtree.ProbitTwoPLNode(person_covariates=True, item_covariates=True),
             irtree.ProbitTwoPLNode(person_covariates=True, item_covariates=True)]
priors_2pl = irtree.build_default_priors(spec, nodes_2pl)
priors_2pl.set("beta",  irtree.NormalPrior(mean=0.0, variance=10.0))
priors_2pl.set("kappa", irtree.NormalPrior(mean=0.0, variance=10.0))
with timed("fit 2PL tree"):
    fit_2pl = irtree.irtree_fit(data, spec, nodes_2pl, priors=priors_2pl,
                                n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED, progress=False)

In [ ]:
summ2 = fit_2pl.summary_df()
slopes = summ2.loc[[n for n in summ2.index if n.startswith("a_")], "mean"]
print("slopes, node 1:", slopes.filter(like="a_1_").round(2).to_numpy())
print("slopes, node 2:", slopes.filter(like="a_2_").round(2).to_numpy())
pd.DataFrame({"Rasch": summ.loc["rho_1_2", ["mean", "sd"]],
              "2PL":   summ2.loc["rho_1_2", ["mean", "sd"]]}).round(3)

## Step 11 — Compare the two trees
Approximate leave-one-out cross-validation (PSIS-LOO), leaving out one person at a time. `compare_loo(first, second)` returns `delta_elpd` = elpd(second) − elpd(first): here positive favours the 2PL tree. Read it against `se_delta`.

In [ ]:
with timed("LOO both models"):
    loo_rasch, loo_2pl = fit_rasch.loo(), fit_2pl.loo()
print("Rasch:", loo_rasch)
print("2PL:  ", loo_2pl)
irtree.compare_loo(loo_rasch, loo_2pl)

## Step 12 — Report (optional)
A self-contained HTML file with the convergence table and trace plots (not committed: `irtree_report*.html` is git-ignored).

In [ ]:
with timed("HTML report"):
    fit_2pl.report("irtree_report_61_2pl.html")

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {"step": "TOTAL", "seconds": rt["seconds"].sum()}
rt.to_csv(RESULTS / "runtime_61.csv", index=False)
rt